# Annotation Validation — MEVA vs Generated Events

This notebook validates generated CCTV event annotations against matched MEVA reference activities.

**Workflow:**
1. Connect to the PostgreSQL `bronze` schema.
2. Load `bronze.matched_pairs`, which already links model-generated events to candidate MEVA references.
3. Identify only rows that have not already been evaluated in `bronze.annotations_report` (incremental/CI-CD input).
4. Use Groq (`openai/gpt-oss-120b`) as an LLM judge for semantic agreement, processing annotations in batches of 20.
5. Build and validate an `annotations_report` DataFrame locally in Kaggle.

This notebook focuses only on **annotation semantic validation**. Temporal IoU and spatial IoU are separate evaluation stages and are not calculated here.


## 1. Environment and database connection

Install the required libraries, load credentials from Kaggle Secrets, and create the SQLAlchemy connection to PostgreSQL. Credentials are not stored directly in the notebook.


In [1]:
%pip install -q pandas sqlalchemy psycopg2-binary groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 33.2 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import re
import pandas as pd

from sqlalchemy import create_engine, text, URL
from groq import Groq
from kaggle_secrets import UserSecretsClient


In [5]:
user_secrets = UserSecretsClient()

DB_PASSWORD = user_secrets.get_secret("DB_PASSWORD")

DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres.uqofqjychtjsvfioebod",
    password=DB_PASSWORD,
    host="aws-0-ap-south-1.pooler.supabase.com",
    port=5432,
    database="postgres",
)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)
print("SQLAlchemy engine created.")


SQLAlchemy engine created.


In [6]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user;")).fetchone()

print("Connection successful.")
print("Database:", result[0])
print("User:", result[1])


Connection successful.
Database: postgres
User: postgres


## 2. Load matched MEVA/model event pairs

`bronze.matched_pairs` is the evaluation input. Each row contains the generated event name/description and the MEVA reference activities selected for that event. The shared `id` is retained as the identifier used by the validation report.


In [7]:
matched_pairs = pd.read_sql(
    "SELECT * FROM bronze.matched_pairs ORDER BY id;",
    engine
)

print("Matched pairs:", len(matched_pairs))
matched_pairs.head()


Matched pairs: 123


,id,event_id,camera_id,model_event_name,model_description,event_start_frame,event_end_frame,meva_reference,n_meva_activities_matched,created_at
0,1,03bb38b8f9bf1abd09a614fef91107eb2a4ea12d11ec43...,G336,Person exits vehicle,A person exits a white SUV as it is parked in ...,1200,1559,a person closes a vehicle door; a person enter...,4,2026-09-16 05:50:14.830168+00:00
1,2,05551c49022f5c5b9ad91131048256fb1940b57f3118d4...,G340,Person appears,A person labeled as 'person #1056' appears in ...,2400,2759,a person opens a trunk; a vehicle stops,2,2026-09-16 05:50:14.830168+00:00
2,3,05e8ccf96167bd997826759d8544542d068e006f5ff1e3...,G340,Vehicle Movement,"A white SUV, labeled as car #2344, moves acros...",5406,5536,(no matching MEVA ground truth found),0,2026-09-16 05:50:14.830168+00:00
3,4,0946514fb83dc5dae4c3f6fcb44c972028f4141750475a...,G340,Car enters frame,A dark-colored sedan enters the frame from the...,8400,8759,a vehicle turns left,1,2026-09-16 05:50:14.830168+00:00
4,5,0bc1db8e2b14bc11ce8011bb7efbe3851426326a167d9a...,G423,Person standing still,"A person is standing still in the hallway, not...",2844,3025,a person exits the scene through a structure,1,2026-09-16 05:50:14.830168+00:00


## 3. Incremental CI/CD input

For future automated runs, only newly added `matched_pairs` rows should be judged. If `bronze.annotations_report` already exists, the query selects IDs that are not yet present in the report. If the report table does not exist yet, all current matched pairs are treated as pending.

This makes the notebook logic incremental and prevents already-evaluated events from being sent to the LLM again.


In [8]:
report_exists_query = """
SELECT EXISTS (
    SELECT 1
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
      AND table_name = 'annotations_report'
);
"""

report_exists = pd.read_sql(report_exists_query, engine).iloc[0, 0]

if report_exists:
    pending_query = """
    SELECT mp.*
    FROM bronze.matched_pairs mp
    LEFT JOIN bronze.annotations_report ar
        ON mp.id = ar.id
    WHERE ar.id IS NULL
    ORDER BY mp.id;
    """
    pending_pairs = pd.read_sql(pending_query, engine)
else:
    pending_pairs = matched_pairs.copy()

print("Annotations pending evaluation:", len(pending_pairs))
pending_pairs.head()


Annotations pending evaluation: 0


,id,event_id,camera_id,model_event_name,model_description,event_start_frame,event_end_frame,meva_reference,n_meva_activities_matched,created_at


## 4. Configure the Groq LLM judge

The Groq API key is loaded from the Kaggle Secret named `groq`. The judge evaluates semantic agreement only; it does not judge temporal overlap or object geometry.


In [9]:
GROQ_API_KEY = user_secrets.get_secret("groq")
groq_client = Groq(api_key=GROQ_API_KEY)

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Reply exactly: Groq connection working"}],
    temperature=0
)

print(response.choices[0].message.content)


Groq connection working


## 5. Batch semantic judge

Annotations are sent in batches of 20. The rubric scores whether each generated annotation is explicitly or semantically supported by its MEVA reference. A score of `0.75` or `1.00` is treated as a semantic match.


In [10]:
import json

def judge_batch(batch_df):
    annotations = []

    for _, row in batch_df.iterrows():
        annotations.append({
            "id": int(row["id"]),
            "model_event_name": row["model_event_name"],
            "model_description": row["model_description"],
            "meva_reference": row["meva_reference"]
        })

    prompt = f"""
You are a strict evaluator comparing generated CCTV annotations
against MEVA reference annotations.

Evaluate each annotation independently.

Do not assume or infer actions that are not explicitly stated.
Related or logically plausible activities are not necessarily the same activity.

Scoring:
1.00 = MEVA explicitly states the same action as the generated event
0.75 = MEVA describes the same core action using different wording
0.50 = generated action is plausible from MEVA, but not explicitly described
0.25 = only weakly related
0.00 = unrelated or contradictory

MATCH is true only when score >= 0.75.

ANNOTATIONS:
{json.dumps(annotations, indent=2)}

Return ONLY a valid JSON array in this format:

[
  {{
    "id": 1,
    "llm_score": 0.75,
    "llm_match": true,
    "llm_reason": "One short sentence."
  }}
]
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content


## 6. Parse LLM output

The judge is instructed to return JSON. This helper removes optional Markdown fences and converts the response into Python records.


In [11]:
def parse_batch_response(response_text):
    cleaned = response_text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return json.loads(cleaned)


## 7. Run pending annotations in batches of 20

Only `pending_pairs` is processed. On the first run this will contain all matched pairs; in later CI/CD runs it will contain only new IDs not already present in `bronze.annotations_report`.


In [12]:
all_judgements = []
batch_size = 20

for start in range(0, len(pending_pairs), batch_size):
    end = min(start + batch_size, len(pending_pairs))
    batch = pending_pairs.iloc[start:end]

    print(f"Processing rows {start + 1}-{end}...")

    response = judge_batch(batch)
    parsed = parse_batch_response(response)
    all_judgements.extend(parsed)

print(f"Total judgments: {len(all_judgements)}")


Total judgments: 0


## 8. Build the local annotation report

Merge the LLM judgments back onto the original pending rows using `id`. The resulting `annotations_report` DataFrame is the local validation output. This cell does **not** write to PostgreSQL.


In [13]:
all_judgements_df = pd.DataFrame(all_judgements)

if len(pending_pairs) == 0:
    annotations_report = pending_pairs.copy()
    annotations_report["llm_score"] = pd.Series(dtype="float")
    annotations_report["llm_match"] = pd.Series(dtype="bool")
    annotations_report["llm_reason"] = pd.Series(dtype="object")
else:
    annotations_report = pending_pairs.merge(
        all_judgements_df,
        on="id",
        how="left"
    )

print("Pending pairs:", len(pending_pairs))
print("Report rows:", len(annotations_report))
print("Missing judgments:", annotations_report["llm_score"].isna().sum())


Pending pairs: 0
Report rows: 0
Missing judgments: 0


In [14]:
annotations_report[
    [
        "id",
        "model_event_name",
        "model_description",
        "meva_reference",
        "llm_score",
        "llm_match",
        "llm_reason"
    ]
].head(20)


,id,model_event_name,model_description,meva_reference,llm_score,llm_match,llm_reason


## 9. Database write — intentionally disabled

The report is kept local until its contents are reviewed. Once approved, a separate write step can append `annotations_report` to `bronze.annotations_report`. Keeping the write disabled prevents accidental database changes during validation and notebook testing.


In [15]:
# Build report only from newly evaluated annotations
if pending_pairs.empty:
    print("No new annotations to write to database.")

else:
    new_judgements_df = pd.DataFrame(all_judgements)

    new_annotations_report = pending_pairs.merge(
        new_judgements_df,
        on="id",
        how="inner"
    )

    new_annotations_report.to_sql(
        "annotations_report",
        engine,
        schema="bronze",
        if_exists="append",
        index=False,
        method="multi"
    )

    print(
        f"Successfully added {len(new_annotations_report)} "
        "rows to bronze.annotations_report."
    )

No new annotations to write to database.


In [16]:
report_count = pd.read_sql(
    """
    SELECT COUNT(*) AS total
    FROM bronze.annotations_report;
    """,
    engine
)

print("Total annotations in report:", report_count.iloc[0]["total"])

Total annotations in report: 123
